In [14]:
from pathlib import Path
datadir = Path("/data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/J1_BXLS_deactivation/test")

In [15]:
import tqdm
from utilities import activity_dict2tensor, ground_truth_rtf_stream
import torch

for scene in datadir.glob("*.pt"):
    data = torch.load(scene, weights_only=False)
    sad_samples = data["meta"]["sad_samples"]
    segments = data["meta"]["segments"]
    rtfs = data["meta"]["rtfs"]
    sad_frames = data["meta"]["sad_frames"]
    gt_rtf_stream, gt_ids_stream, id_map = ground_truth_rtf_stream(
                sad_frames, rtfs, segments
            )
    _, sad_samples_tensor, _, seg_borders = activity_dict2tensor(
        sad_samples, id_map
    )
    Kseg_old = 0
    for start, end in zip(seg_borders[:-1], seg_borders[1:]):
        Nseg = end - start
        
        Kseg = sad_samples_tensor[:-1, start:end].any(dim=-1).sum(dim=0)
        if Kseg > Kseg_old:
            segid = f"A{Kseg}"
        elif Kseg < Kseg_old:
            segid = f"D{Kseg}"
        else:
            segid = f"S{Kseg}"
        Kseg_old = Kseg
        
        if Nseg < 8000:
            print(
                f"Scenario ID: {data['meta']['scenario_id']}, Segment ID: {segid}, Start: {start}, End: {end}, Length: {Nseg} samples"
            )            

Scenario ID: J1_BXLS_deactivation_test_generator_17, Segment ID: A3, Start: 393305, End: 401219, Length: 7914 samples
